<a href="https://colab.research.google.com/github/Asritha0507/ML-Market-Basket-Analysis/blob/main/01_Data_Understanding_and_Exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 1 — Data Understanding and Dataset Scale

## Scalable and Leakage-Free Next-Basket Recommendation

### Objective

This notebook focuses on understanding the complete Instacart Market Basket Analysis dataset before performing feature engineering or model training.

The main objectives are:

- Understand the structure and scale of the available datasets.
- Identify the number of users, orders, products, and purchase interactions.
- Analyze customer ordering behavior and temporal patterns.
- Examine the distribution of reordered and non-reordered products.
- Understand the size of the historical interaction data.
- Establish the foundation for scalable and leakage-free feature engineering.

The complete historical interaction dataset contains millions of purchase records. Therefore, sampling and memory-efficient data loading will be used where appropriate.

In [ ]:
import pandas as pd
import numpy as np
import os
import gc

##  Import Required Libraries

The following libraries are used for data loading, numerical analysis, file handling, and memory management.

- **Pandas** — Data loading and manipulation.
- **NumPy** — Numerical operations.
- **OS** — File and directory inspection.
- **GC** — Explicit memory cleanup when working with large datasets.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
base_path = "/content/drive/MyDrive/ML_Market_Basket_Analysis/Datasets"

print(os.listdir(base_path))

['departments.csv', 'orders.csv', 'products.csv', 'aisles.csv', 'order_products_prior.csv', 'order_products_train.csv', 'Model_Outputs', 'Features', 'Candidate_Training_Data', 'Engineered_Features', 'Augmented_Candidate_Data_Corrected', 'Augmented_Labeled_Candidates_Fixed', 'Augmented_Training_Data', 'Augmented_Candidate_Data', 'Augmented_Labeled_Candidates']


##  Define the Dataset Directory

The dataset directory is defined once so that all files can be accessed consistently throughout the project.

This also makes the notebook easier to maintain because file paths do not need to be repeatedly written in individual cells.

In [ ]:
files = [
    "aisles.csv",
    "departments.csv",
    "orders.csv",
    "products.csv",
    "order_products_prior.csv",
    "order_products_train.csv"
]

for file in files:
    print(file)

aisles.csv
departments.csv
orders.csv
products.csv
order_products_prior.csv
order_products_train.csv


## Identify the Available Datasets

The Instacart dataset is divided into multiple files, each representing a different aspect of the shopping process.

The main files used in this project are:

- `orders.csv` — Information about customer orders.
- `order_products_prior.csv` — Products purchased in previous orders.
- `order_products_train.csv` — Products associated with the training orders.
- `products.csv` — Product information.
- `aisles.csv` — Product aisle information.
- `departments.csv` — Product department information.

First, we verify that all required files are available in the project directory.

In [ ]:
for file in files:
    path = f"{base_path}/{file}"
    print(f"{file}: {os.path.getsize(path) / (1024**2):.2f} MB")

aisles.csv: 0.00 MB
departments.csv: 0.00 MB
orders.csv: 103.92 MB
products.csv: 2.07 MB
order_products_prior.csv: 550.80 MB
order_products_train.csv: 23.54 MB


##  Examine Dataset File Sizes

Before loading the data, we examine the size of each file.

This is important because the historical order-product interaction data contains tens of millions of records. Loading every dataset into memory simultaneously can create memory and scalability problems.

Understanding file sizes helps us design a memory-efficient data processing strategy.

In [ ]:
aisles = pd.read_csv(
    f"{base_path}/aisles.csv"
)

departments = pd.read_csv(
    f"{base_path}/departments.csv"
)

products = pd.read_csv(
    f"{base_path}/products.csv"
)

print("Aisles:", aisles.shape)
print("Departments:", departments.shape)
print("Products:", products.shape)

Aisles: (134, 2)
Departments: (21, 2)
Products: (49688, 4)


##  Load Product and Category Information

The product, aisle, and department datasets are relatively small compared with the historical transaction data.

Therefore, they can be loaded directly into memory.

These datasets will later help us understand the product hierarchy and can also be used for recommendation-related analysis.

In [ ]:
aisles.head()


,aisle_id,aisle
0,1,prepared soups salads
1,2,specialty cheeses
2,3,energy granola bars
3,4,instant foods
4,5,marinades meat preparation


In [ ]:
departments.head()

,department_id,department
0,1,frozen
1,2,other
2,3,bakery
3,4,produce
4,5,alcohol


In [ ]:
products.head()

,product_id,product_name,aisle_id,department_id
0,1,Chocolate Sandwich Cookies,61,19
1,2,All-Seasons Salt,104,13
2,3,Robust Golden Unsweetened Oolong Tea,94,7
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1
4,5,Green Chile Anytime Sauce,5,13


##  Inspect Product and Category Data

The first few records of each dataset are displayed to understand their structure and the relationship between products, aisles, and departments.

In [ ]:
orders = pd.read_csv(
    f"{base_path}/orders.csv",
    usecols=[
        "order_id",
        "user_id",
        "eval_set",
        "order_number",
        "order_dow",
        "order_hour_of_day",
        "days_since_prior_order"
    ],
    dtype={
        "order_id": "int32",
        "user_id": "int32",
        "eval_set": "category",
        "order_number": "int16",
        "order_dow": "int8",
        "order_hour_of_day": "int8",
        "days_since_prior_order": "float32"
    }
)

print(orders.shape)
orders.head()

(3421083, 7)


,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,2539329,1,prior,1,2,8,NaN
1,2398795,1,prior,2,3,7,15.0
2,473747,1,prior,3,3,12,21.0
3,2254736,1,prior,4,4,7,29.0
4,431534,1,prior,5,4,15,28.0


## Load Order-Level Information

The `orders.csv` file contains information about customer ordering behavior.

Only the columns required for our analysis are loaded:

- `order_id` — Unique order identifier.
- `user_id` — Customer identifier.
- `eval_set` — Dataset partition associated with the order.
- `order_number` — Sequential order number for each customer.
- `order_dow` — Day of the week on which the order was placed.
- `order_hour_of_day` — Hour at which the order was placed.
- `days_since_prior_order` — Number of days since the customer's previous order.

Appropriate data types are specified to reduce memory consumption.

In [ ]:
print("Unique users:", orders["user_id"].nunique())
print("Unique orders:", orders["order_id"].nunique())

Unique users: 206209
Unique orders: 3421083


## Determine the Number of Users and Orders

We calculate the number of unique customers and orders in the dataset.

These values provide an initial understanding of the scale of the customer-order relationship that will be used in later feature engineering.

In [ ]:
print(orders["eval_set"].value_counts())

eval_set
prior    3214874
train     131209
test       75000
Name: count, dtype: int64


##  Examine the Dataset Partitions

The `eval_set` column identifies how each order is categorized in the original dataset.

Examining its distribution helps us understand the available historical orders and the original training/testing structure.

For our final project, we will later create a **temporal validation and testing strategy** rather than relying blindly on the original dataset split.

In [ ]:
print("Minimum order number:", orders["order_number"].min())
print("Maximum order number:", orders["order_number"].max())

Minimum order number: 1
Maximum order number: 100


## Analyze Customer Order Sequence

`order_number` represents the chronological position of an order for a particular customer.

This variable is especially important for our project because next-basket recommendation is fundamentally a **temporal prediction problem**.

The model should use a customer's previous behavior to predict a future basket rather than using information from the future.

In [ ]:
print(
    orders["order_number"].describe()
)

count    3.421083e+06
mean     1.715486e+01
std      1.773316e+01
min      1.000000e+00
25%      5.000000e+00
50%      1.100000e+01
75%      2.300000e+01
max      1.000000e+02
Name: order_number, dtype: float64


## Distribution of Customer Order Numbers

The descriptive statistics of `order_number` provide an overview of how many purchasing occasions customers typically have in the dataset.

In [ ]:
print(
    orders["order_dow"].value_counts().sort_index()
)

order_dow
0    600905
1    587478
2    467260
3    436972
4    426339
5    453368
6    448761
Name: count, dtype: int64


##  Ordering Behavior by Day of the Week

The `order_dow` feature represents the day of the week on which an order was placed.

Analyzing this distribution helps identify recurring weekly purchasing patterns that may contribute to customer behavior and recommendation quality.

In [ ]:
print(
    orders["order_hour_of_day"].value_counts().sort_index()
)

order_hour_of_day
0      22758
1      12398
2       7539
3       5474
4       5527
5       9569
6      30529
7      91868
8     178201
9     257812
10    288418
11    284728
12    272841
13    277999
14    283042
15    283639
16    272553
17    228795
18    182912
19    140569
20    104292
21     78109
22     61468
23     40043
Name: count, dtype: int64


##  Ordering Behavior by Hour

The `order_hour_of_day` feature represents the hour at which an order was placed.

Understanding this distribution helps identify common shopping times and provides useful temporal information for later feature engineering.

In [ ]:
prior_path = f"{base_path}/order_products_prior.csv"

with open(prior_path, "r") as f:
    row_count = sum(1 for _ in f) - 1

print("Prior interaction rows:", row_count)

Prior interaction rows: 32434489


##  Determine the Scale of Historical Product Interactions

The `order_products_prior.csv` file contains the products purchased in customers' previous orders.

This is the largest interaction dataset in the project and contains tens of millions of user-product interactions.

Instead of immediately loading the entire file into memory, we first determine its approximate number of records.

This allows us to design a scalable processing strategy for the later feature-engineering stage.

In [ ]:
train_path = f"{base_path}/order_products_train.csv"

with open(train_path, "r") as f:
    train_row_count = sum(1 for _ in f) - 1

print("Train interaction rows:", train_row_count)

Train interaction rows: 1384617


##  Determine the Size of the Training Interaction Dataset

The `order_products_train.csv` file contains the product interactions associated with the original training orders.

We count its records separately to understand the difference between the large historical dataset and the original training interactions.

In [ ]:
prior_sample = pd.read_csv(
    prior_path,
    nrows=100000
)

print(prior_sample.shape)
prior_sample.head()

(100000, 4)


,order_id,product_id,add_to_cart_order,reordered
0,2,33120,1,1
1,2,28985,2,1
2,2,9327,3,0
3,2,45918,4,1
4,2,30035,5,0


##  Inspect a Sample of Historical Interactions

The complete historical interaction dataset is very large.

Therefore, we load only the first 100,000 records for initial inspection rather than loading the entire dataset into memory.

Sampling allows us to understand:

- The structure of the interaction data.
- The relationship between orders and products.
- The `reordered` target variable.
- The approximate characteristics of historical purchases.

Later, we will process the complete dataset using memory-efficient and chunk-based techniques.

In [ ]:
prior_sample.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 4 columns):
 #   Column             Non-Null Count   Dtype
---  ------             --------------   -----
 0   order_id           100000 non-null  int64
 1   product_id         100000 non-null  int64
 2   add_to_cart_order  100000 non-null  int64
 3   reordered          100000 non-null  int64
dtypes: int64(4)
memory usage: 3.1 MB


##  Inspect Data Types and Memory Usage

We examine the data types and memory consumption of the sampled interaction dataset.

This is useful for deciding which data types and processing strategies should be used when working with the complete historical dataset.

In [ ]:
print(
    prior_sample["reordered"].value_counts()
)

reordered
1    59514
0    40486
Name: count, dtype: int64


## Analyze Reorder Behavior

The `reordered` column indicates whether a product was purchased previously by the same customer.

- `0` — Product was not previously reordered.
- `1` — Product was reordered.

Understanding this distribution is important because reorder prediction will be one component of our final recommendation system.

In [ ]:
print(
    prior_sample["reordered"].value_counts(
        normalize=True
    )
)

reordered
1    0.59514
0    0.40486
Name: proportion, dtype: float64


##Reorder Class Distribution

We calculate the proportion of reordered and non-reordered interactions.

This helps us identify whether the target variable is balanced or imbalanced and will guide the choice of evaluation metrics and modelling strategies later.

In [ ]:
print(
    "Unique products in sample:",
    prior_sample["product_id"].nunique()
)

print(
    "Unique orders in sample:",
    prior_sample["order_id"].nunique()
)

Unique products in sample: 16319
Unique orders in sample: 9966


##  Examine Product and Order Diversity

We measure the number of unique products and orders represented in the sampled interaction data.

This provides an initial view of the size of the product space and the number of customer purchase events represented by the interaction dataset.

In [ ]:
print(
    "Orders memory usage:",
    orders.memory_usage(deep=True).sum() / (1024**2),
    "MB"
)

Orders memory usage: 55.46456527709961 MB


##  Measure Memory Usage of the Orders Dataset

Since the final project will work with tens of millions of interaction records, memory efficiency is an important part of the implementation.

We measure the memory consumed by the currently loaded orders dataset.

In [ ]:
print(
    "Prior sample memory usage:",
    prior_sample.memory_usage(deep=True).sum() / (1024**2),
    "MB"
)

Prior sample memory usage: 3.0518836975097656 MB


##  Measure Memory Usage of the Interaction Sample

We also measure the memory consumed by the 100,000-row interaction sample.

This provides a simple comparison between the size of a manageable sample and the much larger complete historical interaction dataset.

In [ ]:
del prior_sample

gc.collect()

0

## Release Temporary Memory

The interaction sample was used only for exploratory analysis.

It is no longer required, so it is deleted and garbage collection is triggered to release memory before continuing with later processing.

#  Notebook Summary

This notebook established the scale and structure of the Instacart dataset.

### Key observations

- The dataset contains millions of customer orders and tens of millions of product interactions.
- Customer purchasing behavior has a strong temporal structure through order sequence, day, hour, and time between orders.
- Reordering is an important behavioral signal and will be used as one component of the recommendation problem.
- The historical interaction data is substantially larger than the original training interaction data.
- Loading the complete interaction dataset directly into memory is unnecessary and can create scalability issues.

### Direction for the next notebook

The next stage will focus on **data preparation and temporal splitting**.

Instead of randomly splitting individual rows, we will construct a time-aware framework in which:

**past customer behavior → predicts future purchasing behavior**

This is essential for preventing data leakage and making the evaluation representative of a real recommendation system.